# Athena 03 — Convert Yelp Reviews CSV to Parquet

Parquet is a columnar format that makes downstream Athena queries (used in EDA and feature engineering) dramatically faster and cheaper than scanning the raw CSV. This notebook uses a CTAS (`CREATE TABLE AS SELECT`) statement to produce a Parquet copy of the `reviews_raw` table under `s3://<bucket>/processed/reviews_parquet/`.

In [ ]:
import boto3
import pandas as pd
from pyathena import connect

%store -r bucket
%store -r region
%store -r database_name
%store -r s3_staging_dir

parquet_table_name = "reviews_parquet"
processed_prefix = "processed/reviews_parquet"
parquet_location = f"s3://{bucket}/{processed_prefix}/"

print("Database:               ", database_name)
print("Parquet table to create:", parquet_table_name)
print("Parquet S3 location:    ", parquet_location)

%store parquet_table_name
%store parquet_location
%store processed_prefix

conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

## Clean up any prior Parquet output

CTAS will fail if its target location already has data. Drop the table and clear the prefix first.

In [ ]:
pd.read_sql(f"DROP TABLE IF EXISTS {database_name}.{parquet_table_name}", conn)

s3 = boto3.client("s3", region_name=region)
paginator = s3.get_paginator("list_objects_v2")
to_delete = []
for page in paginator.paginate(Bucket=bucket, Prefix=processed_prefix):
    for obj in page.get("Contents", []):
        to_delete.append({"Key": obj["Key"]})
if to_delete:
    for i in range(0, len(to_delete), 1000):
        s3.delete_objects(Bucket=bucket, Delete={"Objects": to_delete[i : i + 1000]})
    print(f"Deleted {len(to_delete)} prior Parquet objects")
else:
    print("No prior Parquet objects to delete")

In [ ]:
ctas_statement = f"""
CREATE TABLE {database_name}.{parquet_table_name}
WITH (
    format = 'PARQUET',
    parquet_compression = 'SNAPPY',
    external_location = '{parquet_location}'
) AS
SELECT
    review_id,
    business_id,
    user_id,
    stars,
    review_text,
    date,
    LENGTH(review_text) AS review_char_length
FROM {database_name}.reviews_raw
WHERE review_text IS NOT NULL AND TRIM(review_text) <> ''
"""
print(ctas_statement)
pd.read_sql(ctas_statement, conn)

In [ ]:
summary_df = pd.read_sql(
    f"SELECT COUNT(*) AS row_count, AVG(review_char_length) AS avg_char_length FROM {database_name}.{parquet_table_name}",
    conn,
)
summary_df

## Done

Continue to `../notebooks/02_data_exploration_EDA.ipynb` to perform exploratory data analysis.